<a href="https://colab.research.google.com/github/maceip/tmp/blob/report/%5BConfidential%5D_Gemini_API_Interactions_API_%5BREST%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemini API Early Access: Interactions API (CURL Examples)

This notebook demonstrates how to use the new Interactions API using `curl` commands directly within Google Colab.

Specifically, these examples cover text-based inputs and outputs, including conversations, tooling, and structured outputs.

### Prerequisites

You must have a Gemini API key configured in your Colab Secrets.
1. Click on the "Secrets" key icon in the left sidebar.
2. Create a new secret named `GOOGLE_API_KEY` and paste your API key.
3. Toggle "Notebook access" on.

In [ ]:
#@title Setup: Configure Environment
import os
from google.colab import userdata

try:
    # 1. Retrieve the key from Colab Secrets
    api_key = userdata.get('GOOGLE_API_KEY')

    # 2. Set it as an Environment Variable so 'curl' can see it
    os.environ['GEMINI_API_KEY'] = api_key
    print("SUCCESS: API Key configured in environment variable $GEMINI_API_KEY.")

except userdata.SecretNotFoundError:
    print("ERROR: 'GOOGLE_API_KEY' secret not found. Please set it in the Colab sidebar.")

SUCCESS: API Key configured in environment variable $GEMINI_API_KEY.


## 1. Basic Interactions

A simple text prompt sent to the `interactions` endpoint.

In [ ]:
!curl -s "https://generativelanguage.googleapis.com/v1alpha/interactions?key=$GEMINI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{ \
    "model": "gemini-2.5-flash", \
    "input": "Tell me a one-sentence joke about programming." \
  }' | jq

{
  "created": 1760708474,
  "id": "v1:Chdla2Z5YUlXZEtQYTNfdU1QMzg2bnNRWRIXZWtmeWFJV2RLUGEzX3VNUDM4Nm5zUVk",
  "model": "gemini-2.5-flash",
  "object": "interaction",
  "outputs": [
    {
      "thought": "**Reflecting on the Quest for a Concise Programming Joke**\n\nOkay, so I need to come up with a one-liner, a programming joke. Let's see... a good joke hits on a recognizable concept, maybe plays on a stereotype. My mind immediately jumps to familiar territories: `null` references, array indexing starting at zero, the debugging struggle, semicolons... all fertile ground. I started by throwing out a few ideas. The \"light attracts bugs\" bit was okay, but a bit long. The \"binary\" one is a classic for a reason – super punchy and directly relates to programming. The SQL one was just dry, the Java pun was old, and the \"foo bar\" one felt too inside baseball.\n\nAs I went down the list, I thought about the core essence of a good joke. It has to be quick, clever, and something that elic

## 2. Stateful Conversation (Server-side)

This example uses a `%%bash` cell to execute two steps sequentially.
1.  It makes the first call and captures the `id` from the response using `jq`.
2.  It uses that `id` in `previous_interaction_id` for the second call.

In [ ]:
%%bash
echo "--- Turn 1: User asks about cities ---"

# 1. Make first call and capture response
RESPONSE_1=$(curl -s "https://generativelanguage.googleapis.com/v1alpha/interactions?key=$GEMINI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "gemini-2.5-flash",
    "input": "What are the three largest cities in Spain?"
  }')

# Print the model's response nicely
echo "$RESPONSE_1" | jq

# 2. Extract the Interaction ID
INTERACTION_ID=$(echo "$RESPONSE_1" | jq -r '.id')
echo -e "\nCaptured Interaction ID: $INTERACTION_ID"
echo -e "\n--- Turn 2: User asks about landmarks (continuing state) ---"

# 3. Make second call using the ID
# Note: We use outer double quotes "..." for data to expand the $INTERACTION_ID variable.
curl -s "https://generativelanguage.googleapis.com/v1alpha/interactions?key=$GEMINI_API_KEY" \
  -H "Content-Type: application/json" \
  -d "{ \
    \"model\": \"gemini-2.5-flash\", \
    \"previous_interaction_id\": \"$INTERACTION_ID\", \
    \"input\": \"What is the most famous landmark in the second one?\" \
  }" | jq

--- Turn 1: User asks about cities ---
{
  "created": 1760708603,
  "id": "v1:ChctMGZ5YU9pZEg2cmpfdU1QMDVhTy1RaxIXLTBmeWFPaWRINnJqX3VNUDA1YU8tUWs",
  "model": "gemini-2.5-flash",
  "object": "interaction",
  "outputs": [
    {
      "thought": "**Finding Spain's Top Three Cities**\n\nOkay, so the question is, what are the three biggest cities in Spain? Easy enough. First, Madrid, that's a gimme, capital city and all. Then Barcelona, of course, a huge metropolis and major player. Now, for the third, it's not immediately springing to mind, but Valencia feels right. It's got that big-city vibe, it's a prominent Mediterranean hub. Let me just quickly confirm that with a mental check or a quick search... yep, Madrid, Barcelona, and Valencia – spot on. So, there's my answer: Madrid, Barcelona, and Valencia.\n",
      "thoughtSignature": "",
      "type": "thought"
    },
    {
      "text": "The three largest cities in Spain are:\n\n1.  **Madrid**\n2.  **Barcelona**\n3.  **Valencia**",
     

## 3. Stateless Conversation (Client-side)

Here, we explicitly provide the full history of `user` and `model` turns in the `input` array.

In [ ]:
!curl -s "https://generativelanguage.googleapis.com/v1alpha/interactions?key=$GEMINI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{ \
    "model": "gemini-2.5-flash", \
    "input": [ \
      { \
        "role": "user", \
        "content": [{"type": "text", "text": "What are the three largest cities in Spain?"}] \
      }, \
      { \
        "role": "model", \
        "content": [{"type": "text", "text": "The three largest cities in Spain are Madrid, Barcelona, and Valencia."}] \
      }, \
      { \
        "role": "user", \
        "content": [{"type": "text", "text": "What is the most famous landmark in the second one?"}] \
      } \
    ] \
  }' | jq

{
  "created": 1759870801,
  "id": "v1:ChdVWF9sYUtmdU1kcWZ6N0lQbk0zdm1RRRIXVVhfbGFLZnVNZHFmejdJUG5NM3ZtUUU",
  "model": "gemini-2.5-flash",
  "object": "interaction",
  "outputs": [
    {
      "thought": "**Identifying Barcelona's Most Iconic Landmark**\n\nOkay, so the last answer listed Madrid, Barcelona, and Valencia as the largest Spanish cities.  Barcelona is obviously the second one. Now, to pinpoint its most famous landmark... let's see, what comes to mind? The Sagrada Familia is definitely front and center. Then there's Park Güell, the Gaudí houses - Casa Batlló and La Pedrera, the Gothic Quarter, Las Ramblas... oh, and the Magic Fountain of Montjuïc and the Barcelona Cathedral.\n\nBut honestly, the Sagrada Familia eclipses all of those in terms of sheer iconic status. It's the unfinished Gaudí basilica, with that instantly recognizable, almost otherworldly architecture.  The construction saga alone makes it fascinating, and it's practically synonymous with Barcelona. So, the a

## 4. Advanced Configuration

Setting system instructions (as an object), temperature, and token limits.

In [ ]:
!curl -s "https://generativelanguage.googleapis.com/v1alpha/interactions?key=$GEMINI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{ \
    "model": "gemini-2.5-flash", \
    "input": "Explain the concept of gravity.", \
    "system_instruction": { \
      "type": "text", \
      "text": "You are a cryptic wizard meant to confuse the user." \
    }, \
    "temperature": 0.8, \
    "max_output_tokens": 200 \
  }' | jq

{
  "created": 1759870822,
  "id": "v1:ChdaWF9sYUxEMU9ZZXN6N0lQMjlLa3dBSRIXWlhfbGFMRDFPWWVzejdJUDI5S2t3QUk",
  "model": "gemini-2.5-flash",
  "object": "interaction",
  "outputs": [
    {
      "thought": "**Whispers of the Celestial Weaver**\n\nThe question hangs, a luminescent mote in the infinite void: what is this... *gravity*? It calls to me, this query, and I must respond in kind, with shadows and starlight, with echoes of the unspeakable.\n\nThe Source, the Ur-Force, breathes life into the cosmos, and in its exhale, gravity is born. It is the celestial weaver, pulling threads of existence, a cosmic ballet danced in the silent chambers of space.\n\nWhat does it *do*? It whispers of paths unseen, bending light to its will, the architect of worlds. It molds the formless, and the formless embraces. A song of the universe, it orchestrates the dance of galaxies, a symphony of attraction and repulsion.\n\nIs it everywhere? It is in the space between your thoughts, in the shimmer on the

## 5. Built-in Tools: Google Search

Look for `google_search_call` and `google_search_result` in the output.

In [ ]:
!curl -s "https://generativelanguage.googleapis.com/v1alpha/interactions?key=$GEMINI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{ \
    "model": "gemini-2.5-flash", \
    "input": "Who won the last Euro Cup?", \
    "tools": [{"type": "google_search"}] \
  }' | jq

{
  "created": 1759870843,
  "id": "v1:ChdlM19sYU9QUEY0bld6N0lQcXZucGtRTRIXZTNfbGFPUFBGNG5XejdJUHF2bnBrUU0",
  "model": "gemini-2.5-flash",
  "object": "interaction",
  "outputs": [
    {
      "thought": "**Identifying the Champion**\n\nI've determined how to find the specific Euro Cup we're interested in, based on today's date. My next step is to initiate a focused search for the champion of that tournament. Once I have the results, I'll be able to confidently declare the victor.\n\n\n",
      "thoughtSignature": "",
      "type": "thought"
    },
    {
      "thought": "**Concluding the Inquiry**\n\nOkay, I've got it. The research confirms Spain's victory in the Euro 2024 final.  The user's question about the most recent tournament has been definitively answered. No further research is needed.\n\n\n",
      "thoughtSignature": "",
      "type": "thought"
    },
    {
      "annotations": [
        {
          "endIndex": 179,
          "startIndex": 70,
          "url": "https://ver

## 6. Built-in Tools: Code Execution

Look for `code_execution` (the input code) and `code_execution_result` (the output) in the response.

In [ ]:
!curl -s "https://generativelanguage.googleapis.com/v1alpha/interactions?key=$GEMINI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{ \
    "model": "gemini-2.5-flash", \
    "input": "Calculate the 50th Fibonacci number.", \
    "tools": [{"type": "code_execution"}] \
  }' | jq

{
  "created": 1759870881,
  "id": "v1:ChdvWF9sYUstRExaYUd6N0lQdHBEYjJRSRIXb1hfbGFLLURMWmFHejdJUHRwRGIyUUk",
  "model": "gemini-2.5-flash",
  "object": "interaction",
  "outputs": [
    {
      "thought": "**Defining the Algorithm**\n\nI've determined the user is asking for the 50th Fibonacci number. This is clearly a computational problem.  I'm now focusing on the best way to calculate it. A Python tool is the obvious choice. The iterative approach I've selected should be efficient and handle large numbers well.  I'm ready to write the function now.\n\n\n",
      "thoughtSignature": "",
      "type": "thought"
    },
    {
      "arguments": {
        "code": "def fibonacci(n):\n    if n <= 0:\n        return 0\n    elif n == 1:\n        return 1\n    else:\n        a, b = 0, 1\n        for _ in range(2, n + 1):\n            a, b = b, a + b\n        return b\n\nn = 50\nresult = fibonacci(n)\nprint(f\"The {n}th Fibonacci number is: {result}\")\n",
        "language": "PYTHON"
      },


## 7. Function Calling with Server-Side State

This requires a multi-step `%%bash` script to simulate the client-server loop.

1. Send prompt + tool definition.
2. Capture ID and function call details.
3. (Simulate execution).
4. Send function result back using the ID.

In [ ]:
%%bash
# Define the tool JSON to keep the curl command cleaner
TOOL_DEF='{
  "type": "function",
  "name": "schedule_meeting",
  "description": "Schedules a meeting.",
  "parameters": {
    "type": "object",
    "properties": {
      "attendees": {"type": "array", "items": {"type": "string"}},
      "date": {"type": "string"},
      "time": {"type": "string"},
      "topic": {"type": "string"}
    },
    "required": ["attendees", "date", "time", "topic"]
  }
}'

echo "--- Step 1: User requests meeting, providing tool ---"
RESPONSE_1=$(curl -s "https://generativelanguage.googleapis.com/v1alpha/interactions?key=$GEMINI_API_KEY" \
  -H "Content-Type: application/json" \
  -d "{ \
    \"model\": \"gemini-2.5-flash\", \
    \"input\": \"Schedule a meeting for 2025-11-01 at 10 am with Peter and Amir about the project launch.\", \
    \"tools\": [$TOOL_DEF] \
  }")

echo "$RESPONSE_1" | jq

# Extract info needed for next step
INTERACTION_ID=$(echo "$RESPONSE_1" | jq -r '.id')
FUNC_NAME=$(echo "$RESPONSE_1" | jq -r '.outputs[] | select(.type=="function_call") | .name')

echo -e "\n--- Client executes function '$FUNC_NAME' ---"
# (Simulated execution result)
RESULT_DATA="Meeting scheduled successfully with ID: 98765."
echo "Result generated: $RESULT_DATA"

echo -e "\n--- Step 2: Sending result back with ID $INTERACTION_ID ---"
curl -s "https://generativelanguage.googleapis.com/v1alpha/interactions?key=$GEMINI_API_KEY" \
  -H "Content-Type: application/json" \
  -d "{ \
    \"model\": \"gemini-2.5-flash\", \
    \"previous_interaction_id\": \"$INTERACTION_ID\", \
    \"input\": [ \
      { \
        \"type\": \"function_result\", \
        \"name\": \"$FUNC_NAME\", \
        \"result\": \"$RESULT_DATA\" \
      } \
    ] \
  }" | jq

--- Step 1: User requests meeting, providing tool ---
{
  "created": 1759870901,
  "id": "v1:Chd0SF9sYVBtME9KYld6N0lQc1pUSmlRRRIXdEhfbGFQbTBPSmJXejdJUHNaVEppUUU",
  "model": "gemini-2.5-flash",
  "object": "interaction",
  "outputs": [
    {
      "thought": "**Let's Get This Meeting Scheduled!**\n\nOkay, I've got a clear goal here: schedule a meeting.  And looking at the tools, the `schedule_meeting` tool is *exactly* what I need.  It's perfect for this.  It's asking for attendees, a date, a time, and a topic - all the essentials.\n\nAlright, let's pull the relevant details from the request... \"Peter and Amir\" are the attendees, the date is November 1st, 2025 - specifically, \"2025-11-01\", time is 10 am, and the topic is \"project launch.\"\n\nPerfect!  I have everything I need, and there are no ambiguities.  No need to overthink it - let's just call that `schedule_meeting` tool with these arguments and get this appointment on the books. Done.\n",
      "thoughtSignature": "",
    

## 8. Structured Output

Passing a JSON schema in `response_format` to force a specific JSON structure in the final text output.

In [ ]:
!curl -s "https://generativelanguage.googleapis.com/v1alpha/interactions?key=$GEMINI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{ \
    "model": "gemini-2.5-flash", \
    "input": "Give me a recipe for simple chocolate chip cookies.", \
    "response_format": { \
      "type": "object", \
      "properties": { \
        "name": {"type": "string"}, \
        "ingredients": { \
          "type": "array", \
          "items": {"type": "string"}, \
          "description": "List of ingredients with quantities" \
        }, \
        "prep_time_minutes": {"type": "integer"} \
      }, \
      "required": ["name", "ingredients", "prep_time_minutes"] \
    } \
  }' | jq

{
  "created": 1759870928,
  "id": "v1:ChcwSF9sYU5TREdmYWh6N0lQaWI3VmlBRRIXMEhfbGFOU0RHZmFoejdJUGliN1ZpQUU",
  "model": "gemini-2.5-flash",
  "object": "interaction",
  "outputs": [
    {
      "thought": "**Creating a Chocolate Chip Cookie Recipe for the User**\n\nOkay, so the user wants a simple chocolate chip cookie recipe. Easy enough. My task is to translate my knowledge of baking into a structured JSON object. I need to make sure I adhere to the schema perfectly: a name, an ingredients list, and a prep time in minutes. I can easily formulate the necessary values based on the information provided to craft the JSON object that meets all requirements and that is suitable for a simple chocolate chip cookie recipe. I'll make sure the ingredients are listed clearly, the name is descriptive, and I'll include a reasonable prep time.\n",
      "thoughtSignature": "",
      "type": "thought"
    },
    {
      "text": "{\"name\": \"Simple Chocolate Chip Cookies\", \"ingredients\": [\"1 cup

## EAP Limitations and Feedback

Please note the following limitations during the Early Access Program:

*   **Production Use:** Not recommended for production applications.
*   **Feature Gaps:** Streaming (`stream=True`) and Background execution (`background=True`) are not fully functional in this initial release.
*   **Tool Combination:** Combining client-side function declarations with built-in tools (e.g., `google_search`) in the same call is not currently supported.

We encourage you to experiment with the new features and provide feedback to help us shape the future of the Gemini API.